# PerturbDiff · C2 训练
读取真实训练损失和源域验证回执。

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
fit = json.loads((root / "training_result.json").read_text())
matches = re.findall(
    r"^fit_update=(\d+)/4800 loss=([0-9.e+-]+)$",
    (root / "fit-loss.log").read_text(), re.MULTILINE,
)
updates = [int(step) for step, _ in matches]
loss = [float(value) for _, value in matches]
assert updates == list(range(1, fit["optimizer_updates"] + 1))
assert np.isfinite(loss).all() and fit["test_updates"] == 0
window = 100
smoothed = np.convolve(loss, np.full(window, 1 / window), mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6),
    gridspec_kw={"width_ratios": [3.3, 1, 1]}, layout="constrained")
axes[0].plot(updates, loss, alpha=.18, lw=.8, color="#32758a", label="Training update")
axes[0].plot(updates[window-1:], smoothed, lw=1.6, color="#17666e", label="100-step mean")
axes[0].axhline(fit["validation_loss"], ls="--", color="#ad643d", lw=1.2,
                label=f"Source validation {fit['validation_loss']:.4f}")
axes[0].set(xlabel="Optimizer update", ylabel="Released objective", title="Fit loss")
axes[0].legend(frameon=False, fontsize=8)
axes[1].bar(["Updates"], [fit["optimizer_updates"]], color="#17666e")
axes[1].set_title(f"{fit['optimizer_updates']:,} / {fit['planned_updates']:,}", fontsize=9)
axes[2].bar(["Fit"], [fit["fit_seconds"] / 60], color="#17666e")
axes[2].set(ylabel="Minutes", title=f"{fit['fit_seconds']/60:.2f} min")
for ax in axes:
    ax.grid(axis="y", alpha=.2)
fig.savefig(root / "training-curves.png", dpi=175, bbox_inches="tight")
plt.show()